# Lab 006 — Bootstrap a Sharpe-Ratio Confidence Interval

**Lesson:** [`0006-estimation-inference.html`](../lessons/0006-estimation-inference.html)

**The one skill:** turn a point estimate (a strategy's Sharpe ratio) into an *honest error bar* — a 95% confidence interval — using the **bootstrap**, then see why the i.i.d. version is too optimistic on autocorrelated returns and how a **block bootstrap** fixes it.

**Exit criteria:** the EXIT TICKET cell prints cleanly (all CHECK cells pass).

**How this notebook works**

| Cell tag | You do |
|----------|--------|
| **PROVIDED** | Run it. Imports, data, helpers. |
| **TODO** | Fill the `____` blanks. This is where the learning is. |
| **CHECK** | Run it — immediate assertions. Don't edit. |
| **EXIT TICKET** | Final deliverable. Prints your findings. |

**Environment:** Python 3 + `numpy`, `pandas` (`yfinance` optional). See [`labs/README.md`](./README.md).

## Concept recap (read before coding)

**Annualized Sharpe ratio** of a return series $r$ with per-period mean $\hat\mu$ and std $\hat\sigma$:
$$\widehat{SR} = \frac{\hat\mu}{\hat\sigma}\,\sqrt{P},\qquad P = \text{periods/year (252 for daily)}.$$
It is a *ratio of two estimated, correlated quantities*, so its closed-form standard error is messy — a perfect job for the bootstrap.

**Bootstrap recipe (i.i.d.):**
1. From your $n$ returns, draw $n$ **with replacement** → one bootstrap sample.
2. Compute the statistic (annualized Sharpe) on it.
3. Repeat $B$ times (here $B=10{,}000$).
4. The 2.5th and 97.5th percentiles of those $B$ values are a **95% percentile CI**.

**Lo's i.i.d.-normal formula** (the parametric alternative) gives the per-period Sharpe an approximate standard error $\mathrm{SE}(\widehat{SR}_p)\approx\sqrt{(1+\tfrac12 \widehat{SR}_p^{\,2})/n}$; multiply by $\sqrt{P}$ to annualize. It *assumes i.i.d. (often normal) returns*.

**Block bootstrap:** resample contiguous **blocks** of time (length $L$) instead of single points, so serial dependence (vol clustering, autocorrelation) is preserved. Because dependence shrinks the *effective* sample size, the block CI is usually **wider** — and more honest.

**Toy example (not the lab's answer):** returns `[0.01, -0.02, 0.03]` → $\hat\mu=0.00667$, $\hat\sigma=0.0252$, so per-period Sharpe $\approx 0.264$; annualized (×√252) $\approx 4.2$ — absurdly high precisely because $n=3$ is tiny. The bootstrap CI on such a sample would be enormous, which is the point.

In [ ]:
# PROVIDED — imports, helpers, and data. Tries real data; falls back to a realistic
# simulation with autocorrelation + volatility clustering so the lab runs offline.
import numpy as np
import pandas as pd

ANN = 252  # trading days per year (annualization factor P)

def ann_sharpe(x):
    """Annualized Sharpe ratio of a 1-D array/Series of per-period returns."""
    x = np.asarray(x, dtype=float)
    sd = x.std(ddof=1)
    if sd == 0:
        return 0.0
    return x.mean() / sd * np.sqrt(ANN)

def load_returns(ticker="SPY", n=756, seed=7):
    """Return a pandas Series of daily returns (real if available, else simulated)."""
    try:
        import yfinance as yf  # optional; only if installed + network
        px = yf.download(ticker, period="3y", progress=False)["Close"].dropna()
        r = np.log(px / px.shift(1)).dropna()
        if len(r) > 100:
            print(f"Loaded {len(r)} real daily returns for {ticker}.")
            return pd.Series(np.asarray(r).ravel(), name="ret")
    except Exception as e:
        print(f"(Real data unavailable: {e}) — using a simulated series.")
    # Fallback: AR(1) drift + GARCH-like vol clustering + Student-t (fat) innovations.
    rng = np.random.default_rng(seed)
    omega, alpha, beta, phi = 1e-6, 0.08, 0.90, 0.10
    var = np.empty(n); ret = np.empty(n)
    var[0] = omega / (1 - alpha - beta); ret[0] = 0.0004
    for t in range(1, n):
        var[t] = omega + alpha * ret[t-1]**2 + beta * var[t-1]
        shock = np.sqrt(var[t]) * rng.standard_t(df=5) / np.sqrt(5/3)
        ret[t] = 0.0004 + phi * (ret[t-1] - 0.0004) + shock
    print(f"Simulated {n} daily returns (autocorrelation + vol clustering).")
    return pd.Series(ret, name="ret")

# Separate RNGs so each task is reproducible and independent.
bootrng = np.random.default_rng(1)
blockrng = np.random.default_rng(2)

r = load_returns()
r.describe()

### Task 1 — The point estimate
**Goal:** compute the annualized Sharpe ratio of `r` and store it in `point_sharpe`.

**Why it matters:** this is the number people quote and get excited about — before anyone asks how uncertain it is.

**Hint boundary:** use the provided `ann_sharpe` helper on `r`.

In [ ]:
# TODO — annualized Sharpe ratio of r.
point_sharpe = ____
print(f"Point estimate: annualized Sharpe = {point_sharpe:.3f}")

In [ ]:
# CHECK — do not edit
assert np.isfinite(point_sharpe), "point_sharpe must be a finite number"
assert abs(point_sharpe) < 10, "an annualized Sharpe this large means something is off"
print("Task 1 OK — point Sharpe = %.3f" % point_sharpe)

### Task 2 — The i.i.d. bootstrap CI (the core skill)
**Goal:** resample `r` **with replacement** `B` times, recompute the annualized Sharpe each time, and take the 2.5th/97.5th percentiles as a 95% CI. Store results in `boot` (length `B`) and `ci_iid` (a 2-element array `[lo, hi]`).

**Why it matters:** this is the honest error bar — and it needs *no* formula and *no* normality assumption.

**Hint boundary:** draw indices with `bootrng.integers(0, n, size=n)`; index `vals` with them; use `np.percentile(boot, [2.5, 97.5])`.

In [ ]:
# TODO — i.i.d. bootstrap of the annualized Sharpe.
vals = r.to_numpy()
n = len(vals)
B = 10000
boot = np.empty(B)
for b in range(B):
    idx = ____                       # n indices drawn WITH replacement from 0..n-1
    boot[b] = ann_sharpe(vals[idx])
ci_iid = ____                        # np.percentile(...) for a 95% interval -> [lo, hi]
print(f"i.i.d. bootstrap 95% CI: [{ci_iid[0]:.3f}, {ci_iid[1]:.3f}]  (width {ci_iid[1]-ci_iid[0]:.3f})")

In [ ]:
# CHECK — do not edit
assert boot.shape == (B,), "boot must hold B bootstrap Sharpes"
assert np.all(np.isfinite(boot)), "every bootstrap Sharpe must be finite"
assert ci_iid[0] < ci_iid[1], "CI lower bound must be below the upper bound"
assert ci_iid[0] <= point_sharpe <= ci_iid[1], "the point estimate should sit inside its own bootstrap CI"
print("Task 2 OK — i.i.d. bootstrap CI built")

### Task 3 — Compare to the i.i.d.-normal (Lo) formula
**Goal:** compute Lo's analytic 95% CI assuming i.i.d. returns and store `ci_lo_normal`, `ci_hi_normal`.

**Why it matters:** the parametric interval leans on i.i.d. (often normal) returns. Comparing it to the bootstrap shows how much your conclusion depends on that assumption.

**Hint boundary:** per-period Sharpe `SR_p = point_sharpe / sqrt(ANN)`; `se_p = sqrt((1 + 0.5*SR_p**2)/n)`; annualize with `*sqrt(ANN)`; then `point_sharpe ± 1.96*se_ann`.

In [ ]:
# TODO — Lo's i.i.d.-normal standard error and 95% CI (annualized).
SR_p = ____                          # per-period Sharpe
se_p = ____                          # sqrt((1 + 0.5*SR_p**2)/n)
se_ann = se_p * np.sqrt(ANN)
ci_lo_normal = point_sharpe - 1.96 * se_ann
ci_hi_normal = point_sharpe + 1.96 * se_ann
print(f"i.i.d.-normal 95% CI:   [{ci_lo_normal:.3f}, {ci_hi_normal:.3f}]  (width {ci_hi_normal-ci_lo_normal:.3f})")

In [ ]:
# CHECK — do not edit
assert se_ann > 0, "the standard error must be positive"
assert ci_lo_normal < ci_hi_normal, "CI lower bound must be below the upper bound"
print("Task 3 OK — analytic CI = [%.3f, %.3f]" % (ci_lo_normal, ci_hi_normal))

### Task 4 — The block bootstrap (respect dependence)
**Goal:** resample contiguous **blocks** of length `L` (with replacement) to rebuild a series of length `n`, recompute the Sharpe `B` times, and take a 95% percentile CI as `ci_block`.

**Why it matters:** returns are autocorrelated and volatility clusters, so single-point resampling overstates the effective sample size and the CI is too narrow. Blocks preserve local dependence, giving a more honest (usually wider) interval.

**Hint boundary:** number of blocks `n_blocks = int(np.ceil(n/L))`; draw starts with `blockrng.integers(0, n-L, size=n_blocks)`; build the sample by concatenating `vals[s:s+L]` for each start and trimming to length `n`; then `np.percentile(bootB, [2.5, 97.5])`.

In [ ]:
# TODO — block bootstrap of the annualized Sharpe.
L = 21                                # ~1 trading month per block
n_blocks = int(np.ceil(n / L))
bootB = np.empty(B)
for b in range(B):
    starts = ____                     # n_blocks random block-start indices in 0..n-L-1
    sample = np.concatenate([vals[s:s+L] for s in starts])[:n]
    bootB[b] = ann_sharpe(sample)
ci_block = ____                       # np.percentile(...) for a 95% interval -> [lo, hi]
print(f"block bootstrap 95% CI: [{ci_block[0]:.3f}, {ci_block[1]:.3f}]  (width {ci_block[1]-ci_block[0]:.3f})")

In [ ]:
# CHECK — do not edit
assert bootB.shape == (B,), "bootB must hold B block-bootstrap Sharpes"
assert np.all(np.isfinite(bootB)), "every block-bootstrap Sharpe must be finite"
assert ci_block[0] < ci_block[1], "CI lower bound must be below the upper bound"
print("Task 4 OK — block bootstrap CI built")

In [ ]:
# EXIT TICKET — paste this output to your teacher.
w_iid = ci_iid[1] - ci_iid[0]
w_norm = ci_hi_normal - ci_lo_normal
w_block = ci_block[1] - ci_block[0]
print("=== Sharpe-ratio uncertainty report ===")
print(f"n returns            : {n}")
print(f"point Sharpe (ann.)  : {point_sharpe:+.3f}")
print(f"i.i.d. bootstrap  CI : [{ci_iid[0]:+.3f}, {ci_iid[1]:+.3f}]   width {w_iid:.3f}")
print(f"i.i.d.-normal (Lo) CI: [{ci_lo_normal:+.3f}, {ci_hi_normal:+.3f}]   width {w_norm:.3f}")
print(f"block bootstrap   CI : [{ci_block[0]:+.3f}, {ci_block[1]:+.3f}]   width {w_block:.3f}")
print()
covers_zero = ci_block[0] <= 0 <= ci_block[1]
print(f"Does the block CI include Sharpe = 0? {'YES' if covers_zero else 'no'}")
print("One-sentence takeaway (edit me):")
print("The block-bootstrap CI is the honest one; if it straddles 0, I cannot claim a real edge from this history.")

### Stretch (optional, ungraded)
- Vary the block length `L` (5, 21, 63) and watch the block CI width change — this is the bias/variance trade-off of the block bootstrap itself.
- Add a **BCa** (bias-corrected & accelerated) interval and compare it to the raw percentile CI.
- Re-run on a series with *stronger* autocorrelation (raise `phi` in the simulator) and confirm the block CI widens relative to the i.i.d. one — the dependence penalty made visible.
- Compute the fraction of bootstrap Sharpes below 0 as a rough one-sided "probability the edge is noise" (a frequentist bootstrap p-value cousin — foreshadows Lesson 007).